# Geometry-MMALS G1 - Grounded Functional Geometry

Colab-ready scaffold for the controlled RotatedMNIST experiment. It separates the frozen sensory geometry from context, route, host and synthesis geometry.

> **Status:** implementation scaffold, not a qualified G1 result. No quantum advantage is claimed.

**v1.0.2 numerical-stability patch:** fixes the Fisher-Rao route-loss NaN gradient observed in the v1.0.1 development run, adds finite-value guards and stabilizes the causal tangent probe. C1-C5 remain unqualified.


## 0. Setup

The default clone URL assumes publication under `gharbonnier78/geometry-mmalls-g1`. In a local session, skip the clone and run `pip install -e .`.

In [ ]:
import os
import sys
import shutil
import importlib
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/gharbonnier78/geometry-mmalls-g1.git"
REPO_DIR = Path("/content/geometry-mmalls-g1")
SRC_DIR = REPO_DIR / "src"

# Set FORCE_REFRESH=True after the v1.0.2 package is pushed to GitHub.
FORCE_REFRESH = False
if FORCE_REFRESH and REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

assert (REPO_DIR / "pyproject.toml").exists(), f"Missing pyproject.toml in {REPO_DIR}"
assert (SRC_DIR / "geometry_mmalls" / "__init__.py").exists(), "Missing geometry_mmalls package"

os.chdir(REPO_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(REPO_DIR)],
    check=True,
)

src_path = str(SRC_DIR)
if src_path not in sys.path:
    sys.path.insert(0, src_path)
importlib.invalidate_caches()

import geometry_mmalls
print("Python:", sys.executable)
print("Repository:", REPO_DIR)
print("geometry_mmalls:", geometry_mmalls.__file__)


In [ ]:
from pathlib import Path
import geometry_mmalls

loaded_from = Path(geometry_mmalls.__file__).resolve()
expected_root = Path("/content/geometry-mmalls-g1/src/geometry_mmalls").resolve()
assert expected_root in loaded_from.parents or loaded_from.parent == expected_root, (
    f"Wrong geometry_mmalls package loaded: {loaded_from}"
)
print("Canonical Geometry-G1 package verified:", loaded_from)


In [ ]:
from pathlib import Path
import copy, json, random, time
import numpy as np, pandas as pd, torch, yaml
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from geometry_mmalls.data import FixedAngleMNIST
from geometry_mmalls.geometry import normalized_stress, pairwise_fisher_rao
from geometry_mmalls.metrics import (
    distance_order_correlation,
    euclidean_distance_matrix,
    neighborhood_preservation,
    pairwise_factor_distance,
)
from geometry_mmalls.model import GeometryMMALS, SmallConvEncoder


def route_geodesic_loss(
    routes: torch.Tensor,
    factor: torch.Tensor,
    bandwidth: float = 20.0,
    margin: float = 0.35,
    far_weight: float = 0.25,
    eps: float = 1e-8,
) -> torch.Tensor:
    """Numerically stable G1-A Fisher-Rao route loss.

    The diagonal is masked before arccos and the affinity is clamped with a
    dtype-aware epsilon. This prevents the v1.0.1 finite-forward/NaN-backward
    failure when routes are identical or nearly identical.
    """
    if routes.ndim != 2:
        raise ValueError("routes must have shape [batch, hosts]")
    factor = factor.reshape(-1).to(routes.device, routes.dtype)
    if factor.shape[0] != routes.shape[0]:
        raise ValueError("factor and routes must have the same batch length")

    p = routes.clamp_min(eps)
    p = p / p.sum(dim=-1, keepdim=True)
    roots = torch.sqrt(p)
    affinity = roots @ roots.T
    eye = torch.eye(routes.shape[0], dtype=torch.bool, device=routes.device)
    safe_eps = max(float(eps), 32.0 * torch.finfo(routes.dtype).eps)
    affinity = affinity.masked_fill(eye, 0.0)
    affinity = torch.clamp(affinity, safe_eps, 1.0 - safe_eps)
    d_route = 2.0 * torch.arccos(affinity)
    d_route = d_route.masked_fill(eye, 0.0)

    d_factor = torch.abs(factor[:, None] - factor[None, :])
    near_w = torch.exp(-torch.square(d_factor / max(float(bandwidth), eps)))
    near_w = near_w.masked_fill(eye, 0.0)
    smooth = (near_w * torch.square(d_route)).sum() / near_w.sum().clamp_min(eps)

    far_mask = (d_factor >= float(bandwidth)) & (~eye)
    if torch.any(far_mask):
        far_penalty = torch.relu(float(margin) - d_route[far_mask]).square().mean()
    else:
        far_penalty = torch.zeros((), dtype=routes.dtype, device=routes.device)
    return smooth + float(far_weight) * far_penalty


# Regression gate: v1.0.1 failed here because arccos received affinity == 1.
_probe_logits = torch.zeros(16, 4, requires_grad=True)
_probe_routes = torch.softmax(_probe_logits, dim=-1)
_probe_factor = torch.linspace(-60, 60, 16)
_probe_loss = route_geodesic_loss(_probe_routes, _probe_factor)
_probe_loss.backward()
assert torch.isfinite(_probe_loss)
assert _probe_logits.grad is not None and torch.isfinite(_probe_logits.grad).all()
print("Stable route-geodesic backward gate: PASS")

config = yaml.safe_load(Path('configs/rotated_mnist_g1.yaml').read_text())
RUN_FULL, SEED = False, 0
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(DEVICE, config['data'])


## 1. Controlled data protocol

The true angle is retained for evaluation and declared geometry losses, but is never passed to the deployable router. Interpolation angles remain outside training and model selection.

In [ ]:
root = Path(config['data']['root'])
train_angles = list(map(float, config['data']['train_angles']))
interp_angles = list(map(float, config['data']['interpolation_angles']))
limit = None if RUN_FULL else 2000

def loader(angle, train, shuffle):
    ds = FixedAngleMNIST(root, angle=angle, train=train, download=True)
    if limit: ds = Subset(ds, range(min(limit, len(ds))))
    return DataLoader(ds, batch_size=128, shuffle=shuffle, num_workers=0)

train_loaders = {a: loader(a, True, True) for a in train_angles}
test_loaders = {a: loader(a, False, False) for a in train_angles}
interp_loaders = {a: loader(a, False, False) for a in interp_angles}

## 2. Pretrain and freeze the sensory grove

This boundary prevents geometry already learned by perception from being attributed automatically to MMALS.

In [ ]:
latent_dim = int(config['model']['latent_dim'])
encoder = SmallConvEncoder(latent_dim).to(DEVICE)
head = torch.nn.Linear(latent_dim, 10).to(DEVICE)
opt = torch.optim.AdamW(
    list(encoder.parameters()) + list(head.parameters()),
    lr=float(config['training']['learning_rate']),
    weight_decay=float(config['training']['weight_decay']),
)

sensory_epochs = int(config['training']['sensory_pretrain_epochs']) if RUN_FULL else 2
sensory_history = []
for epoch in range(sensory_epochs):
    encoder.train(); total = correct = 0; epoch_loss = 0.0
    for x, y, _, _ in train_loaders[0.0]:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = head(encoder(x))
        loss = F.cross_entropy(logits, y)
        if not torch.isfinite(loss):
            raise FloatingPointError("Non-finite sensory loss")
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(list(encoder.parameters()) + list(head.parameters()), 5.0)
        opt.step()
        total += y.numel(); correct += (logits.argmax(1) == y).sum().item()
        epoch_loss += float(loss.detach()) * y.numel()
    row = {'epoch': epoch, 'loss': epoch_loss/total, 'accuracy': correct/total}
    sensory_history.append(row)
    print(row)

sensory_state = copy.deepcopy(encoder.state_dict())


## 3. MMALS route-host model and visible G1 losses

In [ ]:
encoder.load_state_dict(sensory_state)
model = GeometryMMALS(
    encoder, latent_dim=latent_dim,
    context_dim=int(config['model']['context_dim']),
    num_hosts=int(config['model']['num_hosts']),
    host_hidden_dim=int(config['model']['host_hidden_dim']),
    freeze_encoder=True,
).to(DEVICE)
opt = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=float(config['training']['learning_rate']),
    weight_decay=float(config['training']['weight_decay']),
)


def diversity(host_outputs):
    h = F.normalize(host_outputs, dim=-1)
    sim = torch.einsum('bhd,bjd->bhj', h, h)
    mask = ~torch.eye(sim.shape[-1], dtype=torch.bool, device=sim.device)
    return sim[:, mask].square().mean()


def assert_finite_trace(tr, where):
    tensors = {
        'z0': tr.z0,
        'context': tr.context,
        'route': tr.route,
        'host_outputs': tr.host_outputs,
        'z_mm': tr.z_mm,
        'logits': tr.logits,
    }
    bad = [name for name, value in tensors.items() if not torch.isfinite(value).all()]
    if bad:
        raise FloatingPointError(f"Non-finite tensors at {where}: {bad}")


def train_stage(angle, epochs=1):
    model.train()
    totals = {'loss': 0.0, 'ce': 0.0, 'geo': 0.0, 'diversity': 0.0, 'samples': 0}
    for epoch in range(epochs):
        for batch_id, (x, y, u, _) in enumerate(train_loaders[angle]):
            x, y, u = x.to(DEVICE), y.to(DEVICE), u.to(DEVICE)
            tr = model(x)
            assert_finite_trace(tr, f"angle={angle}, epoch={epoch}, batch={batch_id}, forward")
            ce = F.cross_entropy(tr.logits, y)
            # G1-A supervised/grounded variant: u shapes the declared geometry
            # loss only. It is never passed to the deployable router.
            geo = route_geodesic_loss(
                tr.route,
                u,
                bandwidth=float(config['losses']['route_bandwidth_degrees']),
                margin=float(config['losses']['route_far_margin']),
            )
            div = diversity(tr.host_outputs)
            loss = (
                float(config['losses']['classification']) * ce
                + float(config['losses']['route_geometry']) * geo
                + float(config['losses']['host_functional_diversity']) * div
            )
            if not torch.isfinite(torch.stack([ce, geo, div, loss])).all():
                raise FloatingPointError(
                    f"Non-finite loss at angle={angle}, epoch={epoch}, batch={batch_id}: "
                    f"ce={ce.item()}, geo={geo.item()}, diversity={div.item()}, loss={loss.item()}"
                )
            opt.zero_grad(set_to_none=True)
            loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 5.0
            )
            if not torch.isfinite(torch.as_tensor(grad_norm)):
                raise FloatingPointError(f"Non-finite gradient norm at angle={angle}, batch={batch_id}")
            opt.step()

            n = y.numel()
            totals['samples'] += n
            totals['loss'] += float(loss.detach()) * n
            totals['ce'] += float(ce.detach()) * n
            totals['geo'] += float(geo.detach()) * n
            totals['diversity'] += float(div.detach()) * n

    n = max(totals.pop('samples'), 1)
    return {'angle': angle, **{k: v/n for k, v in totals.items()}}


## 4. Continual sequence

For qualification, add hardened EWC, replay, sparse-MoE, oracle-angle diagnostic and joint upper-bound comparisons with equal validation budgets.

In [ ]:
stage_logs, checkpoints = [], {}
stage_epochs = int(config['training']['stage_epochs']) if RUN_FULL else 2
for angle in train_angles:
    row = train_stage(angle, stage_epochs)
    stage_logs.append(row)
    checkpoints[angle] = copy.deepcopy(model.state_dict())
    print(row)
stage_logs


## 5. Trace collection and geometry evidence

In [ ]:
@torch.no_grad()
def collect(loaders, max_batches=3):
    model.eval(); rows=[]
    for angle, dl in loaders.items():
        for b, (x, y, u, idx) in enumerate(dl):
            if max_batches is not None and b >= max_batches:
                break
            tr = model(x.to(DEVICE))
            assert_finite_trace(tr, f"collect angle={angle}, batch={b}")
            for i in range(len(y)):
                rows.append(dict(
                    angle=float(u[i]),
                    label=int(y[i]),
                    prediction=int(tr.logits[i].argmax()),
                    z0=tr.z0[i].cpu().numpy(),
                    context=tr.context[i].cpu().numpy(),
                    route=tr.route[i].cpu().numpy(),
                    z_mm=tr.z_mm[i].cpu().numpy(),
                    hosts=tr.host_outputs[i].cpu().numpy(),
                ))
    return pd.DataFrame(rows)

trace = collect({**test_loaders, **interp_loaders}, None if RUN_FULL else 3)
assert len(trace) > 0
print("trace rows:", len(trace))


In [ ]:
stack = lambda name: np.stack(trace[name].to_numpy())
for name in ['z0', 'context', 'route', 'z_mm']:
    assert np.isfinite(stack(name)).all(), f"Non-finite values in trace space: {name}"

du = pairwise_factor_distance(trace.angle.to_numpy())
spaces = {
    'sensory': euclidean_distance_matrix(stack('z0')),
    'context': euclidean_distance_matrix(stack('context')),
    'route': pairwise_fisher_rao(stack('route')),
    'synthesis': euclidean_distance_matrix(stack('z_mm')),
}
rows = []
for name, d in spaces.items():
    assert np.isfinite(d).all(), f"Non-finite distance matrix: {name}"
    rho, p = distance_order_correlation(du, d)
    rows.append(dict(
        space=name,
        rho=rho,
        p_value=p,
        stress=normalized_stress(du, d),
        knn=neighborhood_preservation(du, d, k=5),
        status='development_not_qualified',
    ))
geometry_df = pd.DataFrame(rows)
geometry_df


## 6. Host ablation evidence

A host is not specialized merely because it receives route mass. Measure the loss of true-class evidence when it is removed and the remaining route is renormalized.

In [ ]:
@torch.no_grad()
def ablations(loaders, max_batches=2):
    model.eval(); rows=[]
    for angle, dl in loaders.items():
        for b, (x, y, _, _) in enumerate(dl):
            if b >= max_batches:
                break
            x, y = x.to(DEVICE), y.to(DEVICE)
            tr = model(x)
            assert_finite_trace(tr, f"ablation angle={angle}, batch={b}")
            base = F.log_softmax(tr.logits, -1).gather(1, y[:, None]).squeeze(1)
            for h in range(tr.route.shape[1]):
                r = tr.route.clone()
                r[:, h] = 0
                r = r / r.sum(1, keepdim=True).clamp_min(1e-8)
                z = torch.einsum('bh,bhd->bd', r, tr.host_outputs)
                out = model.classifier(model.synthesis_norm(z))
                impact = base - F.log_softmax(out, -1).gather(1, y[:, None]).squeeze(1)
                if not torch.isfinite(impact).all():
                    raise FloatingPointError(f"Non-finite ablation impact: angle={angle}, host={h}")
                rows += [
                    dict(angle=angle, host=h, ablation_impact=float(v), status='development_not_qualified')
                    for v in impact.cpu()
                ]
    return pd.DataFrame(rows)

ablation_df = ablations(test_loaders)
ablation_df.groupby(['angle', 'host']).ablation_impact.mean().unstack()


## 7. Minimal causal tangent probe

The qualified experiment must add matched-norm orthogonal controls, bootstrap confidence intervals and identity-preservation gates.

In [ ]:
trained = trace[trace.angle.isin(train_angles)]
Cmat = np.stack(trained.context).astype(np.float64)
y = trained.angle.to_numpy(float)
assert np.isfinite(Cmat).all() and np.isfinite(y).all()

X = Cmat - Cmat.mean(0, keepdims=True)
y0 = y - y.mean()
ridge = 1e-6 * max(np.trace(X.T @ X) / max(X.shape[1], 1), 1.0)
tangent = np.linalg.solve(X.T @ X + ridge*np.eye(X.shape[1]), X.T @ y0)
tangent_norm = np.linalg.norm(tangent)
if not np.isfinite(tangent_norm) or tangent_norm < 1e-12:
    raise FloatingPointError("Causal tangent is undefined or degenerate")
tangent /= tangent_norm

@torch.no_grad()
def reroute(z0, c):
    return torch.softmax(model.router(torch.cat([z0, c], -1)), -1)

x = next(iter(interp_loaders[15.0]))[0].to(DEVICE)
tr = model(x)
assert_finite_trace(tr, "causal probe")
t = torch.tensor(tangent, dtype=tr.context.dtype, device=DEVICE)
causal_probe_df = pd.DataFrame([
    {
        'scale': s,
        'route_shift': float(torch.linalg.vector_norm(
            reroute(tr.z0, tr.context + s*t) - tr.route,
            dim=1,
        ).mean()),
        'status': 'development_not_qualified',
    }
    for s in [-2, -1, 0, 1, 2]
])
causal_probe_df


## 8. Export with epistemic status

In [ ]:
out = Path('results/per_seed/dev_seed_0')
out.mkdir(parents=True, exist_ok=True)
geometry_df.to_csv(out/'geometry_metrics.csv', index=False)
ablation_df.to_csv(out/'host_ablation.csv', index=False)
causal_probe_df.to_csv(out/'causal_probe.csv', index=False)
pd.DataFrame(stage_logs).to_csv(out/'stage_logs.csv', index=False)
pd.DataFrame(sensory_history).to_csv(out/'sensory_history.csv', index=False)
(out/'run_manifest.json').write_text(json.dumps({
    'status': 'development_not_qualified',
    'notebook_version': '1.0.2',
    'seed': SEED,
    'run_full': RUN_FULL,
    'device': str(DEVICE),
    'timestamp': time.time(),
    'warning': 'Not a qualified G1 result.',
}, indent=2))
print(out.resolve())


## 9. Before publication and optional PDF export

Implement transport memory, dual-memory qualification, cross-seed host-role matching, compute accounting, confidence intervals, baseline hardening and secondary-dataset replication. Follow `docs/CLAIMS_AND_GATES.md`.

The cells below are optional. Save the notebook to Drive or upload it to `/content` before running the PDF export.


In [ ]:
# Optional Drive mount. Skip when the notebook already exists under /content.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print("Drive mount skipped:", exc)


In [ ]:
# Optional one-time LaTeX installation for nbconvert PDF export.
import shutil, subprocess, sys
if shutil.which("xelatex") is None:
    subprocess.run(
        ["apt-get", "update"],
        check=True,
        stdout=subprocess.DEVNULL,
    )
    subprocess.run(
        ["apt-get", "install", "-y", "texlive-xetex", "texlive-latex-extra", "pandoc"],
        check=True,
    )
else:
    print("xelatex already installed:", shutil.which("xelatex"))


In [ ]:
from pathlib import Path

patterns = [
    "Geometry_MMALS_G1_Colab_v1_0_2*.ipynb",
    "Geometry_MMALS_G1_Colab_v1_0_1*.ipynb",
]
candidates = []
for pattern in patterns:
    candidates.extend(Path('/content').glob(pattern))
    drive_root = Path('/content/drive/MyDrive')
    if drive_root.exists():
        candidates.extend(drive_root.rglob(pattern))

# Deduplicate and prefer the newest file.
candidates = sorted({p.resolve() for p in candidates if p.is_file()}, key=lambda p: p.stat().st_mtime, reverse=True)
if not candidates:
    raise FileNotFoundError(
        "Notebook file not found. Save a copy to Drive or upload the .ipynb to /content, then rerun this cell."
    )
notebook_path = candidates[0]
print("Notebook selected for export:", notebook_path)


In [ ]:
import subprocess, sys
from pathlib import Path

output_dir = Path('/content/pdf_export')
output_dir.mkdir(parents=True, exist_ok=True)
cmd = [
    sys.executable, '-m', 'jupyter', 'nbconvert',
    '--to', 'pdf',
    str(notebook_path),
    '--output-dir', str(output_dir),
    '--PDFExporter.latex_command=xelatex',
    '--PDFExporter.latex_command={filename}',
    '--PDFExporter.latex_command=-interaction=nonstopmode',
    '--PDFExporter.latex_count=3',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)
print("PDF export directory:", output_dir)
